# 01 — Ingestion audit (Phase 0)

Читает **только готовые output-файлы** из `data/processed/model_inputs/` —
никакого production parsing здесь нет.

Label sources (`data/processed/label_sources/`) **намеренно не анализируются**
в этом notebook, чтобы не смешивать model inputs и label sources.


In [1]:
import json
from pathlib import Path

import pandas as pd

REPO_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
MODEL_INPUTS = REPO_ROOT / 'data' / 'processed' / 'model_inputs'
assert MODEL_INPUTS.is_dir(), (
    f'нет {MODEL_INPUTS}; сначала выполните dalel ingest'
)
print('Читаем только:', MODEL_INPUTS)

Читаем только: /Users/izbassar/Documents/Projects/GovTech-Camp-2026/data/processed/model_inputs


In [2]:
def read_json(path: Path):
    return json.loads(path.read_text(encoding='utf-8'))

def read_jsonl(path: Path):
    if not path.is_file():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

documents = []
reports = []
for document_json in sorted(MODEL_INPUTS.glob('*/*/document.json')):
    documents.append(read_json(document_json))
    report_path = document_json.parent / 'ingestion_report.json'
    if report_path.is_file():
        reports.append(read_json(report_path))

docs = pd.DataFrame(documents)
reps = pd.DataFrame(reports)
print(f'Проектов на диске: {docs.project_id.nunique() if len(docs) else 0}')
print(f'Обработанных документов: {len(docs)}')
docs[['project_id', 'document_id', 'document_type', 'role', 'extraction_status',
      'parser_name', 'page_count', 'document_mode']] if len(docs) else docs

Проектов на диске: 4
Обработанных документов: 19


,project_id,document_id,document_type,role,extraction_status,parser_name,page_count,document_mode
0,project_001_bereke,project_001_bereke__action_plan__001,action_plan,model_input,success,docling,2,scanned
1,project_001_bereke,project_001_bereke__ndv__001,ndv,model_input,success,docling,143,mixed
2,project_001_bereke,project_001_bereke__nontechnical_summary__001,nontechnical_summary,model_input,success,docling,6,digital
3,project_001_bereke,project_001_bereke__pek__001,pek,model_input,success,docling,31,mixed
4,project_001_bereke,project_001_bereke__puo__001,puo,model_input,success,docling,56,mixed
5,project_002_azm,project_002_azm__action_plan__001,action_plan,model_input,success,docling,2,digital
6,project_002_azm,project_002_azm__ndv__001,ndv,model_input,partial,docling,174,mixed
7,project_002_azm,project_002_azm__nontechnical_summary__001,nontechnical_summary,model_input,success,docling,1,docx_flow
8,project_002_azm,project_002_azm__pek__001,pek,model_input,success,docling,20,digital
9,project_002_azm,project_002_azm__puo__001,puo,model_input,success,docling,31,digital


## Распределения: document_type, role, статусы

In [3]:
if len(docs):
    display(docs.document_type.value_counts().rename('по document_type').to_frame())
    display(docs.role.value_counts().rename('по role').to_frame())
    display(docs.extraction_status.value_counts().rename('по статусу').to_frame())
    success_like = docs.extraction_status.isin(['success', 'partial']).sum()
    print(f'Parse success rate (success+partial): {success_like}/{len(docs)}'
          f' = {success_like / len(docs):.0%}')

,по document_type
document_type,
action_plan,3
ndv,3
nontechnical_summary,3
pek,3
puo,3
roos,2
explanatory_note,1
working_project_note,1


,по role
role,
model_input,19


,по статусу
extraction_status,
success,15
partial,4


Parse success rate (success+partial): 19/19 = 100%


## Страницы и режимы PDF (digital / scanned / mixed)

In [4]:
if len(docs):
    display(docs.document_mode.value_counts().rename('document_mode').to_frame())
    print('Всего страниц:', int(docs.page_count.fillna(0).sum()))
    display(docs[['document_id', 'document_mode', 'page_count']]
            .sort_values('page_count', ascending=False).head(10))

,document_mode
document_mode,
digital,10
mixed,7
scanned,1
docx_flow,1


Всего страниц: 1044


,document_id,document_mode,page_count
6,project_002_azm__ndv__001,mixed,174
18,project_004_sintez_ural__working_project_note_...,digital,159
11,project_003_bayterek__roos__001,mixed,144
1,project_001_bereke__ndv__001,mixed,143
13,project_004_sintez_ural__ndv__001,mixed,135
17,project_004_sintez_ural__roos__001,digital,60
4,project_001_bereke__puo__001,mixed,56
16,project_004_sintez_ural__puo__001,mixed,36
9,project_002_azm__puo__001,digital,31
3,project_001_bereke__pek__001,mixed,31


## OCR usage

In [5]:
if len(docs):
    ocr = pd.json_normalize(docs['ocr'])
    ocr.insert(0, 'document_id', docs.document_id.values)
    display(ocr[['document_id', 'mode', 'engine', 'engine_ran',
                 'ocr_page_count', 'candidate_pages', 'warnings']])
    print('Документов с реально выполненным OCR:', int(ocr.engine_ran.sum()))

,document_id,mode,engine,engine_ran,ocr_page_count,candidate_pages,warnings
0,project_001_bereke__action_plan__001,auto,easyocr,True,2,"[1, 2]",[]
1,project_001_bereke__ndv__001,auto,easyocr,True,15,"[2, 123, 125, 126, 127, 128, 130, 131, 132, 13...",[]
2,project_001_bereke__nontechnical_summary__001,auto,NaN,False,0,[],[]
3,project_001_bereke__pek__001,auto,easyocr,True,2,"[1, 2]",[]
4,project_001_bereke__puo__001,auto,easyocr,True,5,"[1, 2, 44, 55, 56]",[]
5,project_002_azm__action_plan__001,auto,NaN,False,0,[],[]
6,project_002_azm__ndv__001,auto,easyocr,True,0,"[26, 136]",[ocr_language_unsupported: kk (easyocr runs ru...
7,project_002_azm__nontechnical_summary__001,auto,NaN,False,0,[],[]
8,project_002_azm__pek__001,auto,NaN,False,0,[],[]
9,project_002_azm__puo__001,auto,NaN,False,0,[],[]


Документов с реально выполненным OCR: 8


## Таблицы, изображения, секции, fallback

In [6]:
if len(reps):
    summary = reps[['document_id', 'pages_processed', 'ocr_pages', 'table_count',
                    'image_count', 'section_count', 'warning_count',
                    'fallback_used', 'elapsed_seconds', 'hash_unchanged']]
    display(summary)
    print('Всего сериализованных валидных таблиц:', int(reps.table_count.sum()))
    if 'skipped_empty_table_items' in reps:
        skipped = reps['skipped_empty_table_items'].fillna(0).astype(int)
        print('Пропущено пустых table items (не сериализованы):', int(skipped.sum()))
        nonzero = reps.loc[skipped > 0, 'document_id'].tolist()
        print('  документы с пропусками:', nonzero or 'нет')
    else:
        print('Пропущено пустых table items: поле отсутствует в отчётах (schema < 1.1.0)')
    print('Всего изображений:', int(reps.image_count.sum()))
    print('Fallback использован:', int(reps.fallback_used.sum()), 'раз(а)')
    assert bool(reps.hash_unchanged.all()), 'raw hash изменился — расследовать немедленно!'

,document_id,pages_processed,ocr_pages,table_count,image_count,section_count,warning_count,fallback_used,elapsed_seconds,hash_unchanged
0,project_001_bereke__action_plan__001,2,2,1,3,3,0,False,20.934,True
1,project_001_bereke__ndv__001,143,15,90,92,278,0,False,470.126,True
2,project_001_bereke__nontechnical_summary__001,6,0,1,0,5,0,False,15.038,True
3,project_001_bereke__pek__001,31,2,16,10,70,0,False,241.343,True
4,project_001_bereke__puo__001,56,5,30,52,99,0,False,447.940,True
5,project_002_azm__action_plan__001,2,0,2,1,1,0,False,85.308,True
6,project_002_azm__ndv__001,174,0,134,72,329,2,False,360.045,True
7,project_002_azm__nontechnical_summary__001,1,0,0,1,1,1,False,5.466,True
8,project_002_azm__pek__001,20,0,18,1,25,0,False,93.403,True
9,project_002_azm__puo__001,31,0,10,2,32,0,False,24.749,True


Всего сериализованных валидных таблиц: 623
Пропущено пустых table items (не сериализованы): 54
  документы с пропусками: ['project_004_sintez_ural__ndv__001', 'project_004_sintez_ural__roos__001']
Всего изображений: 460
Fallback использован: 0 раз(а)


## Warnings

In [7]:
from collections import Counter
warning_counter = Counter()
for record in documents:
    for warning in record.get('warnings', []):
        warning_counter[warning[:100]] += 1
pd.DataFrame(warning_counter.most_common(20), columns=['warning', 'count'])

,warning,count
0,ocr_language_unsupported: kk (easyocr runs ru+en),2
1,"pages without text after OCR policy: [26, 136]",1
2,docx flow document exposes no reliable page nu...,1
3,"pages without text after OCR policy: [20, 21, ...",1
4,"pages without text after OCR policy: [67, 69, ...",1
5,empty_table_item_skipped: document_id=project_...,1
6,empty_table_item_skipped: document_id=project_...,1
7,empty_table_item_skipped: document_id=project_...,1
8,pages without text after OCR policy: [34],1
9,empty_table_item_skipped: document_id=project_...,1


## Примеры текста и таблиц

In [8]:
if len(docs):
    sample_dir = MODEL_INPUTS / docs.iloc[0].project_id / docs.iloc[0].document_id
    pages = read_jsonl(sample_dir / 'pages.jsonl')
    if pages:
        page = pages[0]
        print(f"{docs.iloc[0].document_id} — страница {page['page_number']}"
              f" ({page['char_count']} симв., ocr={page['ocr_applied']}):")
        print(page['text'][:1200])

project_001_bereke__action_plan__001 — страница 1 (111 симв., ocr=True):
1
3
I
=
g
3
1
I 3 1 1
= 1 5 3 5 1 5 ;
Роспубпихасы
%
Ig
3
$
1
1
Шыгыс
стан
1
' ekcem
Scanned with
Cs CamScanner


In [9]:
shown = False
for document_json in sorted(MODEL_INPUTS.glob('*/*/tables.jsonl')):
    tables = read_jsonl(document_json)
    if tables:
        table = max(tables, key=lambda t: t['num_rows'] * max(t['num_cols'], 1))
        print(f"{table['table_id']} — {table['num_rows']}x{table['num_cols']},"
              f" стр. {table['page_number']}")
        display(pd.DataFrame(table['cells']).head(8))
        shown = True
        break
if not shown:
    print('Таблиц пока нет')

project_001_bereke__action_plan__001__tab_0001 — 17x3, стр. 1


,0,1,2
0,,,'
1,1 = ? 3 3,I1 '; 3 1 1 8 8 1 1 1 Il 8 3,
2,Ili 1 1,Il 1 3,5 1
3,,,Il
4,1,1 1 g 5 3,
5,,,1
6,,Ё,5 2 ;
7,,,#


## Документы с плохим extraction coverage

In [10]:
rows = []
for document_json in sorted(MODEL_INPUTS.glob('*/*/document.json')):
    record = read_json(document_json)
    pages = read_jsonl(document_json.parent / 'pages.jsonl')
    if not pages:
        continue
    empty_pages = sum(1 for p in pages if p['char_count'] < 32)
    rows.append({
        'document_id': record['document_id'],
        'status': record['extraction_status'],
        'pages': len(pages),
        'empty_pages': empty_pages,
        'empty_ratio': round(empty_pages / len(pages), 2),
        'median_chars': int(pd.Series([p['char_count'] for p in pages]).median()),
    })
coverage = pd.DataFrame(rows).sort_values('empty_ratio', ascending=False) if rows else None
coverage

,document_id,status,pages,empty_pages,empty_ratio,median_chars
5,project_002_azm__action_plan__001,success,2,1,0.50,267
13,project_004_sintez_ural__ndv__001,partial,135,27,0.20,248
10,project_003_bayterek__explanatory_note__001,success,12,2,0.17,2118
4,project_001_bereke__puo__001,success,56,9,0.16,1688
16,project_004_sintez_ural__puo__001,partial,36,3,0.08,3032
11,project_003_bayterek__roos__001,partial,144,9,0.06,1950
18,project_004_sintez_ural__working_project_note_...,success,159,5,0.03,2852
17,project_004_sintez_ural__roos__001,success,60,2,0.03,84
3,project_001_bereke__pek__001,success,31,1,0.03,2002
1,project_001_bereke__ndv__001,success,143,4,0.03,748


---
**Напоминание о leakage boundary:** этот notebook смотрит только на
`data/processed/model_inputs/`. Никогда не подключайте сюда
`label_sources/` — post-review документы не должны попадать в аналитические
признаки и датасеты моделей.
